# 非程式題

1. 如果 Agent 沒有 planning 能力會有什麼缺點？

2. planning 在實務應用上會碰到什麼困難？

# 程式題

In [17]:
!pip install --quiet -U langgraph langchain-openai langchain-community tavily-python

zsh:1: command not found: pip


## 將下列的小工具加進你的 plan-and-execute agent 裡面，使得輸出類似的結果

> 小小 hint：tutorial 照摳就好好🙂，或是大家可以自己寫一些 function 或是找一些外部 api 工具，玩一下看看有什麼新花樣

In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:

from langchain_core.tools import tool
import datetime
from langchain_community.tools.tavily_search import TavilySearchResults

@tool
def get_current_date() -> str:
    """取得現在的日期，格式為 YYYY-MM-DD"""
    return datetime.datetime.now().strftime("%Y-%m-%d")

In [20]:
import requests
import pandas as pd
from langchain_core.tools import tool

@tool
def getMainBuyAndSale() :
  """台積電股價及其他相關資訊"""
  url = f'https://pchome.megatime.com.tw/stock/sto1/ock4/sid' + '2330' + '.html'
  payloads = {'is_check': '1'}
  resp = requests.request(method='POST', url=url, data=payloads)
  data = pd.read_html(resp.text)

  return data

In [21]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
executor = create_react_agent(llm, tools=[getMainBuyAndSale, get_current_date], prompt='你是一個專業等級的任務執行器，用來完美執行任務計劃。')

In [22]:
import operator
from typing import Annotated, List, Tuple
from typing_extensions import TypedDict
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class PlanExeute(TypedDict):
    input: str
    plan: List[str]
    past_steps: Annotated[List[str], operator.add]
    response: str
    
class Plan(BaseModel):
    """未來要遵循的計劃"""

    steps: List[str] = Field(description="不同的步驟，應按順序排列")

planner_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            你是一個專業等級的任務規劃器。
            請根據指定目標，擬定一份簡潔的逐步執行計畫。
            此計畫應包含一系列獨立的任務，每個任務若正確執行，將逐步導向正確的答案。
            請避免加入任何多餘的步驟。
            最終步驟的執行結果即為最終答案。
            請確保每個步驟皆包含完成所需的所有資訊，不得跳過任何必要步驟。
            """,
        ),
        ("placeholder", "{messages}"),
    ]
) 

planner = planner_prompt | ChatOpenAI(
    model="gpt-4.1-2025-04-14",
    temperature=0
).with_structured_output(Plan)

In [23]:
from typing import Union


class Response(BaseModel):
    """回應使用者的內容"""

    response: str


class Act(BaseModel):
    """要執行的動作"""

    action: Union[Response, Plan] = Field(
        description="要執行的動作： 若要回應使用者，請使用'Response'。"
        "若需進一步使用工具以取得答案，請使用'Plan'。"
    )

replanner_prompt = ChatPromptTemplate.from_template(
    """
    你是一個專業等級的任務規劃器。
    請根據指定目標，擬定一份簡潔的逐步執行計畫。
    此計畫應包含一系列獨立的任務，每個任務若正確執行，將逐步導向正確的答案。
    請避免加入任何多餘的步驟。
    最終步驟的執行結果即為最終答案。
    請確保每個步驟皆包含完成所需的所有資訊，不得跳過任何必要步驟。

    本次任務的目標為：
    {input}

    你原先擬定的計畫為：
    {plan}

    目前已執行的步驟如下：
    {past_steps}

    請根據目前進度，調整後續的執行計畫。
    若已無需進一步步驟，並可直接回應使用者，請進行回應；
    若仍需執行工具或任務取得答案，請補足剩餘計畫。
    請僅列出尚未完成且仍需執行的步驟，勿重複列出已執行的部分。
    """
)


replanner = replanner_prompt | ChatOpenAI(
    model="gpt-4.1-2025-04-14",
    temperature=0
).with_structured_output(Act)

In [24]:
from langgraph.graph import END


# 執行計劃中的第一個步驟
async def execute_step(state: PlanExeute):
    plan = state["plan"]
    # 將所有步驟編號整理成文字（用於提示）
    plan_str = "\n".join(f"{i+1}. {step}" for i, step in enumerate(plan))
    task = plan[0] # 抓出第一個尚未執行的步驟
    task_formatted = f"""針對以下計劃：{plan_str}\n\n你的任務是執行 step {1}, {task}."""

    # 呼叫執行器執行該步驟
    agent_response = await executor.ainvoke(
        {"messages": [("user", task_formatted)]}
    )

    # 回傳本次執行結果（步驟與回應）
    return {
        "past_steps": [(task, agent_response["messages"][-1].content)],
    }

# 初始規劃流程
async def plan_step(state: PlanExeute):
    plan = await planner.ainvoke({"messages": [("user", state["input"])]})
    return {"plan": plan.steps}

# 根據已完成的步驟與整體輸入，更新後續步驟或直接產生答案。
async def replan_step(state: PlanExeute):
    output = await replanner.ainvoke(state)

    # 若模型決定可以直接回覆使用者（任務結束），則回傳結果
    if isinstance(output.action, Response):
        return {"response": output.action.response}
    else:
        # 否則更新剩餘計劃
        return {"plan": output.action.steps}

# 檢查是否已經產生最終回應，決定是否結束流程
def should_end(state: PlanExeute):
    if "response" in state and state["response"]:
        return END
    else:
        return "executor"
     

In [25]:
from langgraph.graph import StateGraph, START

graph_builder = StateGraph(PlanExeute)

graph_builder.add_node("planner", plan_step) # 新增規劃器節點
graph_builder.add_node("executor", execute_step) # 新增執行器節點
graph_builder.add_node("replanner", replan_step) # 新增重新規劃器節點

graph_builder.add_edge(START, "planner")
graph_builder.add_edge("planner", "executor") # 將規劃器與執行器連接
graph_builder.add_edge("executor", "replanner") # 將執行器與重新規劃器連接

graph_builder.add_conditional_edges(
    "replanner",
    should_end,
    ["executor", END],
)

graph = graph_builder.compile()

In [26]:

from langchain_core.callbacks.base import BaseCallbackHandler

class PrintToolNameHandler(BaseCallbackHandler):
    def on_tool_start(self, serialized_tool: dict, input_str, **kwargs):
        tool_name = serialized_tool.get("name", "")
        print(f"[Tool name] {tool_name}")

In [27]:
config = {"recursion_limit": 50,
          "callbacks":[PrintToolNameHandler()]}
inputs = {"input": "明天適合進場台積電嗎?"}

async for event in graph.astream(inputs, config=config):
    for k, v in event.items():
        if k != "__end__":
            for step in v.keys():
              if step == "plan":
                print(f"[{step}]")
                for idx, action in enumerate(v[step]):
                  print(f"{idx+1}. {action}")

              elif step == "past_steps":
                print(f"[{step}]")
                for action in v[step][0]:
                  print(action)

              else:
                print(f"[{step}]")
                print(v[step])
    print('\n')

[plan]
1. 查詢台積電（2330）最新收盤價及今日盤後消息。
2. 分析台積電近期技術線型（如均線、成交量、K線型態）。
3. 查閱近期台積電相關基本面消息（如財報、產業新聞、法說會重點）。
4. 檢視大盤（加權指數）及半導體產業整體趨勢。
5. 評估市場情緒與外資、投信近期買賣超狀況。
6. 根據上述資訊，判斷明天台積電是否具備進場條件。


[Tool name] getMainBuyAndSale


/var/folders/93/jy5s5h4d6wb954g_5_jtd1m40000gn/T/ipykernel_46405/938373103.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  data = pd.read_html(resp.text)


[Tool name] getMainBuyAndSale


/var/folders/93/jy5s5h4d6wb954g_5_jtd1m40000gn/T/ipykernel_46405/938373103.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  data = pd.read_html(resp.text)


[Tool name] getMainBuyAndSale


/var/folders/93/jy5s5h4d6wb954g_5_jtd1m40000gn/T/ipykernel_46405/938373103.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  data = pd.read_html(resp.text)


[past_steps]
查詢台積電（2330）最新收盤價及今日盤後消息。
目前無法查詢台積電（2330）的最新收盤價及今日盤後消息。請稍後再試，或考慮使用其他方式獲取這些資訊。


[response]
由於無法取得台積電（2330）最新收盤價及今日盤後消息，缺乏關鍵即時資訊，無法進行後續技術面、基本面與市場情緒等分析。因此，無法判斷明天是否適合進場台積電。建議待取得最新市場資訊後再進行評估。


